### Prototipo geocoder

Notebook para prototipar a função de geocodificação de endereços.

A função se baseará na camada de logradouros do GeoSampa (posteriormente podendo ser migrada para as camadas internas do MDSF).

A geocodificação partirá do código do logradouro já identificado anteriormente no APP (posteriormente podemos implementar o fluxo completo, com a identificação do logradouro pelo nome, como um serviço em si mesmo).

O fluxo de processamento de dados é o seguinte:

1. Pelo WFS, obter todos os segmentos de reta relacionados ao CODLOG;
2. Remover os segmentos de reta que não possuem numeração no lado par e no lado ímpar (ocorre em alguns casos em que, por exemplo, uma rotatória foi representada, ou outra estrutura como canteiro central)
3. Identificar a paridade da numeração inputada pelo usuário (se é par ou ímpar)
4. Identificar o segmento de reta que contém a numeração indicada pelo usuário
    - Casoa não encontre retorna mensagem de erro
5. Arrumar a orientação do segmento de reta:
    - Identificar o segmento de reta que tem a numeração (para a paridade) logo antes
        - Caso seja o segmento inicial, tem que inverter a lógica para o segmento logo depois. Isso podemos identificar verificando se o numero inicial é o menor para aquela paridade
    - Verificar se o ponto inicial do segmento de reta de interesse está mais próximo do ponto final do segmento de reta anterior ou do ponto inicial
        - Inverter se for o segmento inicial (aí o ponto final do logradouro tem que estar próximo do inicial do outro - podemos fazer isso espertamente trocando id por ego nos parametros da função)
    - Se estiver mais próximo do ponto final, a orientação está correta, não precisa fazer nada. Se estiver mais próximo do ponto inicial, a orientação está errada, então precisa inverter as coordenadas
6. Interpolar a numeração dentro do segmento de reta ordenado
7. Gerar o ponto
8. Retornar a resposta com metadados (usar um modelo Pydantic)

In [ ]:
from api.integrations.wfs import WFSFetcher
from enum import Enum
from api.config import settings
import geopandas as gpd


class Paridade(Enum):
    IMPAR = 1
    PAR = 2


class SegmentoNotFoundError(Exception):
    pass

class NumeracaoNotFoundError(Exception):
    pass

class DimapGeocoder:

    cols_numeracao = {
        Paridade.PAR: {
        "inicial" :   'cd_numero_inicial_par',
        "final" : 'cd_numero_final_par'     
    },
    Paridade.IMPAR: {
        "inicial" : 'cd_numero_inicial_impar',
        "final" : 'cd_numero_final_impar'
    }
    }

    def __init__(self)->None:

        self.wfs = WFSFetcher()
        self.layer_logradouros = settings.LAYER_LOGRADOUROS

    @property
    def cols_numeracao_lst(self):
        cols = []
        for paridade in Paridade:
            cols.extend(self.cols_numeracao[paridade].values())
        return cols

    def get_segmentos(self, codlog:int)->gpd.GeoDataFrame:

        segmentos = []

        for batch in self.wfs(self.layer_logradouros, cql_filter=f"codlog={codlog}"):
            segmentos.extend(batch)

        gdf_segmentos = gpd.GeoDataFrame.from_features(segmentos)

        if gdf_segmentos.empty:
            #usando erro especifico para poder dar catch depois e retornar 404
            raise SegmentoNotFoundError(f"Nenhum segmento encontrado para codlog={codlog}")

        return gdf_segmentos
    
    def paridade_numeracao(self, numero:int)->Paridade:

        if numero % 2 == 0:
            return Paridade.PAR
        else:
            return Paridade.IMPAR
        
    def clean_segmentos_sem_numeracao(self, gdf_segmentos:gpd.GeoDataFrame)->gpd.GeoDataFrame:

       df_numeracao = gdf_segmentos[self.cols_numeracao_lst]
       sem_numeracao_nenhum_lado = df_numeracao.isnull().all(axis=1)
       return gdf_segmentos[~sem_numeracao_nenhum_lado]
    

    def find_segmento_contem_numero(self, gdf_segmentos:gpd.GeoDataFrame, numero:int)->gpd.GeoDataFrame:

        paridade = self.paridade_numeracao(numero)
        col_inicial = self.cols_numeracao[paridade]["inicial"]
        col_final = self.cols_numeracao[paridade]["final"]

        #filtrando segmentos que contem o numero buscado
        gdf_segmentos_contem_numero = gdf_segmentos[
            (gdf_segmentos[col_inicial] <= numero) & 
            (gdf_segmentos[col_final] >= numero)
        ]

        if gdf_segmentos_contem_numero.empty:
            raise NumeracaoNotFoundError(f"Nenhum segmento encontrado contendo o numero {numero}")

        if gdf_segmentos_contem_numero.shape[0] > 1:
            print(f"Warning: mais de um segmento encontrado contendo o numero {numero}. Retornando o primeiro.")
            gdf_segmentos_contem_numero = gdf_segmentos_contem_numero.iloc[[0]]

        return gdf_segmentos_contem_numero
    
    def is_primeiro_segmento(self, gdf_segmentos:gpd.GeoDataFrame, segmento:gpd.GeoDataFrame, paridade:Paridade)->bool:

        coluna_inicial = self.cols_numeracao[paridade]["inicial"]

        numero_inicial_segmento = segmento.iloc[0][coluna_inicial]

        #menor numero inicial
        menor_numero_inicial = gdf_segmentos[coluna_inicial].min()

        return numero_inicial_segmento == menor_numero_inicial
    
    def is_ultimo_segmento(self, gdf_segmentos:gpd.GeoDataFrame, segmento:gpd.GeoDataFrame, paridade:Paridade)->bool:

        coluna_final = self.cols_numeracao[paridade]["final"]

        numero_final_segmento = segmento.iloc[0][coluna_final]

        #maior numero final
        maior_numero_final = gdf_segmentos[coluna_final].max()

        return numero_final_segmento == maior_numero_final
    

    def segmento_anterior(self, gdf_segmentos:gpd.GeoDataFrame, segmento:gpd.GeoDataFrame, paridade:Paridade)->gpd.GeoDataFrame:

        if self.is_primeiro_segmento(gdf_segmentos, segmento, paridade):
            raise ValueError("Segmento é o primeiro da via, não existe segmento anterior")
        
        coluna_inicial = self.cols_numeracao[paridade]["inicial"]
        coluna_final = self.cols_numeracao[paridade]["final"]
        numero_inicial_segmento = segmento.iloc[0][coluna_inicial]

        #nao deveria ser menor ou igual (só menor) mas estou colocando por precaução
        segmentos_anteriores = gdf_segmentos[gdf_segmentos[coluna_final] <= numero_inicial_segmento] 
        segmentos_anteriores.sort_values(by=coluna_final, ascending=False, inplace=True)
        return segmentos_anteriores.iloc[[0]]
        
    def segmento_posterior(self, gdf_segmentos:gpd.GeoDataFrame, segmento:gpd.GeoDataFrame, paridade:Paridade)->gpd.GeoDataFrame:

        if self.is_ultimo_segmento(gdf_segmentos, segmento, paridade):
            raise ValueError("Segmento é o último da via, não existe segmento posterior")
        coluna_inicial = self.cols_numeracao[paridade]["inicial"]
        coluna_final = self.cols_numeracao[paridade]["final"]
        numero_final_segmento = segmento.iloc[0][coluna_final]

        #nao deveria ser maior ou igual (só maior) mas estou colocando por precaução
        segmentos_posteriores = gdf_segmentos[gdf_segmentos[coluna_inicial] >= numero_final_segmento] 
        segmentos_posteriores.sort_values(by=coluna_inicial, ascending=True, inplace=True)
        return segmentos_posteriores.iloc[[0]]


    def orientacao_segmento(self, segmento:gpd.GeoDataFrame)->str:

        #aqui precisa implementar a checagem dos numeros inicial e final em relacao ao segmento adjacente
        pass


    def pipeline_geocode(self, codlog:int, numero:int)->gpd.GeoDataFrame:

        gdf_segmentos = self.get_segmentos(codlog)
        gdf_segmentos = self.clean_segmentos_sem_numeracao(gdf_segmentos)
        segmento_numero = self.find_segmento_contem_numero(gdf_segmentos, numero)

        return segmento_numero



In [22]:
layer_logradouros = settings.LAYER_LOGRADOUROS

In [23]:
wfs = WFSFetcher()

In [24]:
codlog_paulista = '156566'

In [25]:
segmentos = []

for batch in wfs(layer_logradouros, cql_filter=f"codlog={codlog_paulista}"):
    segmentos.extend(batch)

In [26]:
gdf_segmentos = gpd.GeoDataFrame.from_features(segmentos)

In [29]:
gdf_segmentos.head()

,geometry,cd_identificador,cd_identificador_logradouro,codlog,cd_tipo_logradouro,cd_titulo_logradouro,tx_preposicao_logradouro,nm_logradouro,cd_origem_denominacao,cd_numero_inicial_par,cd_numero_final_par,cd_numero_inicial_impar,cd_numero_final_impar,cd_numero_ordem_segmento,qt_leito_carrocavel
0,"LINESTRING (331870.62 7392526.991, 331724.103 ...",42806,45434,156566,AV,None,None,PAULISTA,None,NaN,NaN,409.0,613.0,7.0,2
1,"LINESTRING (331712.048 7392655.443, 331703.826...",42807,45434,156566,AV,None,None,PAULISTA,None,NaN,NaN,613.0,623.0,9.0,2
2,"LINESTRING (330278.349 7393988.43, 330272.539 ...",42819,45434,156566,AV,None,None,PAULISTA,None,2574.0,2580.0,NaN,NaN,49.0,1
3,"LINESTRING (331151.002 7393166.465, 331082.454...",43234,45434,156566,AV,None,None,PAULISTA,None,1372.0,1462.0,NaN,NaN,19.0,2
4,"LINESTRING (332201.773 7392275.973, 332039.276...",51142,45434,156566,AV,None,None,PAULISTA,None,0.0,292.0,NaN,NaN,1.0,2


In [30]:
gdf_segmentos.columns

Index(['geometry', 'cd_identificador', 'cd_identificador_logradouro', 'codlog',
       'cd_tipo_logradouro', 'cd_titulo_logradouro',
       'tx_preposicao_logradouro', 'nm_logradouro', 'cd_origem_denominacao',
       'cd_numero_inicial_par', 'cd_numero_final_par',
       'cd_numero_inicial_impar', 'cd_numero_final_impar',
       'cd_numero_ordem_segmento', 'qt_leito_carrocavel'],
      dtype='str')